# Подготовка данных в Google Drive

Этот ноутбук качает все исходные биржевые данные (модуль 1 проекта)
и складывает их в общий каталог на Google Drive. После прогона
обучающий ноутбук `training_pipeline.ipynb` будет брать данные
именно оттуда - не понадобится повторно дёргать MOEX ISS / ЦБ РФ /
Yahoo Finance в каждой Colab-сессии.

**Структура на Drive:**
```
/MyDrive/lstm_exchange/
├── data/
│   └── raw/
│       ├── moex/        OHLCV акций (один CSV на тикер)
│       ├── indexes/      IMOEX, RGBI, RTSI
│       └── macro/        курсы ЦБ, ключевая ставка, Brent
└── checkpoints/         сюда обучающий ноутбук пишет артефакт-пакет
```

## 1. Окружение и репозиторий

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    if PROJECT_ROOT.exists():
        print(f'{PROJECT_ROOT} уже существует - пропускаем клонирование.')
    else:
        env = os.environ.copy()
        env['GIT_TERMINAL_PROMPT'] = '0'
        subprocess.run(
            [
                'git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
                f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git',
                str(PROJECT_ROOT),
            ],
            check=True,
            env=env,
        )
        print(f'Репозиторий склонирован в {PROJECT_ROOT}')
else:
    PROJECT_ROOT = Path.cwd().parent

PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'IN_COLAB:     {IN_COLAB}')

In [ ]:
if IN_COLAB:
    print('Устанавливаем зависимости загрузчика данных...')
    subprocess.run(
        [
            'pip', 'install', '--quiet',
            'pandas>=2.1', 'pyarrow>=15.0', 'requests>=2.31',
            'yfinance>=0.2.40', 'tqdm>=4.66', 'ipywidgets>=8.0',
        ],
        check=True,
    )
    print('Готово.')
else:
    print('Локальный режим: используются зависимости из текущего окружения.')


In [ ]:
import sys

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s | %(message)s')

from graduate_work.config import default_config

cfg = default_config()
print(f'Тикеры ({len(cfg.data.tickers)}):', cfg.data.tickers)
print('Период:', cfg.data.start_date, '→', cfg.data.end_date)

## 2. Подключение Google Drive

В Colab монтируем `/content/drive/`. Локально - пропускаем шаг и
используем `data/raw/` рядом с проектом.

In [ ]:
DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DATA_RAW = DRIVE_BASE / 'data' / 'raw'
    DRIVE_DATA_RAW.mkdir(parents=True, exist_ok=True)
    (DRIVE_BASE / 'checkpoints').mkdir(parents=True, exist_ok=True)
    print(f'Drive data dir: {DRIVE_DATA_RAW}')
else:
    DRIVE_DATA_RAW = cfg.paths.data_raw
    print(f'Локально data/raw: {DRIVE_DATA_RAW}')

## 3. Утилиты двусторонней синхронизации

Простая копия дерева: `shutil.copytree` с `dirs_exist_ok=True`. Этого
хватает - суммарный объём всех CSV для 30 тикеров × 4 лет дневных
свечей около 5 МБ.

In [ ]:
def sync_tree(src: Path, dst: Path) -> int:
    """Скопировать всё дерево src → dst, перезаписывая дубликаты. Возвращает число файлов."""
    if not src.exists():
        return 0
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    return sum(1 for _ in dst.rglob('*.csv'))


def list_drive_summary(root: Path) -> None:
    if not root.exists():
        print(f'  {root} (пусто)')
        return
    for sub in sorted(root.iterdir()):
        if sub.is_dir():
            files = sorted(p.name for p in sub.glob('*.csv'))
            print(f'  {sub.name}/ ({len(files)} файлов)')
            if files:
                print('     ' + ', '.join(files[:8]) + ('...' if len(files) > 8 else ''))

## 4. Подтянуть существующие данные с Drive (если есть)

Это полезно при повторных запусках: если на Drive что-то уже лежит,
перенесём в локальный `data/raw/`, чтобы `download_all` пропустил
уже скачанные тикеры.

In [ ]:
if IN_COLAB and DRIVE_DATA_RAW.exists():
    n = sync_tree(DRIVE_DATA_RAW, cfg.paths.data_raw)
    print(f'Подтянули с Drive {n} CSV-файлов в {cfg.paths.data_raw}')
else:
    print('Локальный режим или Drive пуст - пропускаем.')

## 5. Скачивание новых данных

Качаем всё, чего не хватает: акции (MOEX ISS), индексы, макро-ряды
(ЦБ РФ + Yahoo). Уже сохранённые файлы перезапишутся свежей
версией.

In [ ]:
from graduate_work.data.orchestrator import (
    download_indexes, download_macro, download_tickers,
)

# Тикеры - самая медленная часть; качаем порциями.
ticker_results = download_tickers(cfg.data, cfg.paths)
print(f'Скачано/обновлено тикеров: {len(ticker_results)}')

In [ ]:
index_results = download_indexes(cfg.data, cfg.paths)
print(f'Скачано индексов: {len(index_results)} ({list(index_results.keys())})')

In [ ]:
macro_results = download_macro(cfg.data, cfg.paths)
print(f'Скачано макрорядов: {len(macro_results)} ({list(macro_results.keys())})')

## 6. Выгрузить локальные данные на Drive

In [ ]:
if IN_COLAB:
    n = sync_tree(cfg.paths.data_raw, DRIVE_DATA_RAW)
    print(f'Выгрузили на Drive: {n} CSV-файлов в {DRIVE_DATA_RAW}')
else:
    print(f'Локальные данные уже в {cfg.paths.data_raw}')

## 7. Проверка содержимого

In [ ]:
print('=== Drive (data/raw) ===')
if IN_COLAB:
    list_drive_summary(DRIVE_DATA_RAW)
else:
    list_drive_summary(cfg.paths.data_raw)

## Что дальше

- Откройте `training_pipeline.ipynb` - первая ячейка автоматически
  смонтирует Drive и подтянет данные из этой папки в локальный
  `data/raw/`.
- Если нужно обновить только новые тикеры - добавьте их в
  `DataConfig.tickers` и перезапустите этот ноутбук; уже скачанные
  файлы будут переиспользованы.